# Feature Engineering for Wildfire Prediction

---

## 🎯 Purpose of This Notebook

This notebook focuses on **creating and transforming features** to improve model performance for predicting wildfire evacuation zone impact. Feature engineering is often the most impactful step in the machine learning pipeline!

### 💡 Why Feature Engineering Matters

**"Better features beat better algorithms"** - A common data science principle

- **Captures Domain Knowledge**: Encodes wildfire behavior patterns into features
- **Improves Model Performance**: Well-engineered features can dramatically boost accuracy
- **Reduces Complexity**: Good features allow simpler models to work well
- **Handles Data Issues**: Addresses skewness, zeros, outliers, and multicollinearity

### 📋 What We'll Accomplish

## Objectives

1. **Handle zero-inflated features**: Many wildfire features have >50% zeros (sparse activity)
2. **Create interaction features**: Combine features to capture relationships (e.g., distance × speed)
3. **Generate polynomial features**: Capture non-linear relationships
4. **Create domain-specific wildfire features**: Leverage wildfire science knowledge
5. **Address multicollinearity**: Remove redundant, highly correlated features
6. **Feature scaling**: Normalize features for model compatibility
7. **Feature selection**: Keep only the most predictive features
8. **Prepare final feature sets for modeling**: Export clean, ready-to-use datasets

### 🔑 Key Concepts

**Zero-Inflated Features**:
- Features with many zeros (e.g., growth rate = 0 when fire isn't growing)
- Solution: Create binary indicator (is_zero) + log transform of non-zero values

**Interaction Features**:
- Combine two features to capture their joint effect
- Example: `distance × closing_speed` = time until potential impact

**Domain-Specific Features**:
- Features based on wildfire science (e.g., fire intensity, threat velocity)
- Often more predictive than raw measurements

**Multicollinearity**:
- When features are highly correlated (>0.9)
- Problem: Redundant information, unstable models
- Solution: Remove one feature from each correlated pair

## 1. Setup and Data Loading

### 🔧 Libraries and Configuration

We'll use several specialized libraries for feature engineering:
- **sklearn.preprocessing**: Scaling and transformations
- **sklearn.feature_selection**: Selecting best features  
- **statsmodels**: Variance Inflation Factor (VIF) for multicollinearity

In [ ]:
# Core libraries
import pandas as pd  # Data manipulation
import numpy as np  # Numerical operations
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns  # Statistical visualization

# Feature engineering and preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA

# Multicollinearity detection
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Set random seed for reproducibility
# This ensures feature selection and other random operations are consistent
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Libraries loaded successfully")

In [ ]:
# Load data
# We'll engineer features on both train and test sets simultaneously
# This ensures consistency in feature creation
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
metadata_df = pd.read_csv('../data/metaData.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Display target variable information (only in training data)
print(f"\nTarget variables in train:")
print(f"- time_to_hit_hours: {train_df['time_to_hit_hours'].describe()}")
print(f"- event: {train_df['event'].value_counts()}")

In [ ]:
# Separate features and targets
# Target columns are what we're trying to predict (only in train data)
target_cols = ['time_to_hit_hours', 'event']
id_col = 'event_id'

# Get feature columns (everything except targets and ID)
feature_cols = [col for col in train_df.columns if col not in target_cols + [id_col]]

# Create feature matrices and target vector
# .copy() ensures we don't modify original dataframes
X_train = train_df[feature_cols].copy()
y_train = train_df[target_cols].copy()
X_test = test_df[feature_cols].copy()

print(f"Number of features: {len(feature_cols)}")

# Display feature categories from metadata
# This helps us understand the domain organization of features
print(f"\nFeature categories from metadata:")
feature_metadata = metadata_df[metadata_df['type'] == 'feature']
print(feature_metadata['category'].value_counts())
print(f"\n💡 We'll create features that combine information across these categories")

## 2. Handle Zero-Inflated Features

### 🎯 Why Handle Zero-Inflated Features?

**Zero-inflated features** are variables where >50% of values are exactly zero. These are common in wildfire data:
- Fire may not have grown in certain directions
- No precipitation in dry periods
- Zero wind speed in calm conditions

**Problem**: Standard transformations (like log) don't work well with many zeros

**Solution**: Two-part strategy:
1. **Binary indicator**: Create `feature_is_zero` (1 if zero, 0 otherwise)
   - Captures the *presence/absence* pattern
2. **Log transformation**: Apply `log1p(x)` to original feature
   - `log1p(x) = log(1 + x)` handles zeros gracefully
   - Reduces skewness in non-zero values

This gives the model both:
- Whether the feature is zero (categorical information)
- The magnitude when non-zero (continuous information)

In [ ]:
# Identify zero-inflated features (>50% zeros)
# Calculate percentage of zeros for each feature
zero_pct = (X_train == 0).sum() / len(X_train) * 100

# Filter features with >50% zeros
zero_inflated_features = zero_pct[zero_pct > 50].index.tolist()

print(f"Zero-inflated features (>50% zeros): {len(zero_inflated_features)}")
print("\nTop 10 zero-inflated features:")
print(zero_pct[zero_inflated_features].sort_values(ascending=False).head(10))
print(f"\n💡 These features need special handling to preserve information")

In [ ]:
# Create binary indicators for zero-inflated features
def create_zero_indicators(df, features):
    """
    Create binary indicators for zero-inflated features.
    
    For each zero-inflated feature, creates a new binary feature:
    - 1 if original value is zero
    - 0 if original value is non-zero
    
    Example: If 'fire_growth_north' is often zero:
    - 'fire_growth_north_is_zero' = 1 means no growth in that direction
    - This captures important absence information
    """
    df_new = df.copy()
    for feat in features:
        # Create binary indicator: 1 if zero, 0 otherwise
        df_new[f'{feat}_is_zero'] = (df[feat] == 0).astype(int)
    return df_new

# Apply to both train and test sets
X_train = create_zero_indicators(X_train, zero_inflated_features)
X_test = create_zero_indicators(X_test, zero_inflated_features)

print(f"✅ Added {len(zero_inflated_features)} binary indicators")
print(f"New feature count: {X_train.shape[1]}")

In [ ]:
# Apply log1p transformation to positive-valued features
def apply_log_transform(df, features):
    """
    Apply log1p transformation to handle skewness.
    
    log1p(x) = log(1 + x) has several advantages:
    - Handles zeros: log1p(0) = 0 (no undefined values)
    - Reduces right skewness (long tail of large values)
    - Compresses large values while preserving small values
    - Makes distributions more normal-like
    
    Example transformation:
    - Original: [0, 1, 10, 100, 1000]
    - log1p:    [0, 0.69, 2.40, 4.62, 6.91]
    - Notice how large values are compressed
    """
    df_new = df.copy()
    for feat in features:
        # Only transform non-negative features (log requires positive values)
        if df[feat].min() >= 0:
            df_new[f'{feat}_log1p'] = np.log1p(df[feat])
    return df_new

# Apply to zero-inflated features
X_train = apply_log_transform(X_train, zero_inflated_features)
X_test = apply_log_transform(X_test, zero_inflated_features)

print(f"✅ Added {len(zero_inflated_features)} log-transformed features")
print(f"New feature count: {X_train.shape[1]}")
print(f"\n💡 Now we have 3 versions of each zero-inflated feature:")
print(f"   1. Original: Raw values")
print(f"   2. Binary: Is it zero?")
print(f"   3. Log-transformed: Normalized magnitude")

## 3. Create Interaction Features

### 🔥 Why Interaction Features?

**Interaction features** capture relationships between variables that are more predictive together than separately.

**Example**: A fire that is:
- Close to evacuation zone (low distance) AND
- Growing rapidly (high growth rate)
- Is MUCH more dangerous than either factor alone

**Key Wildfire Interactions**:
1. **Distance × Growth**: How fast is threat approaching?
2. **Distance × Speed**: Direct closing velocity
3. **Growth × Speed**: Fire expansion dynamics
4. **Distance × Alignment**: Is fire moving toward zone?
5. **Area ÷ Distance**: Threat density (large fire nearby)

These capture the **physics of wildfire threat** better than individual features.

In [ ]:
# 1. Distance × Growth interaction (critical for prediction)
# Captures: How quickly is the growing fire approaching?
# High value = Fast-growing fire that's close = HIGH THREAT
X_train['dist_growth_interaction'] = X_train['dist_min_ci_0_5h'] * X_train['area_growth_rate_ha_per_h']
X_test['dist_growth_interaction'] = X_test['dist_min_ci_0_5h'] * X_test['area_growth_rate_ha_per_h']

# 2. Distance × Speed interaction
# Captures: Direct closing velocity toward evacuation zone
# Positive closing_speed means fire is approaching
X_train['dist_speed_interaction'] = X_train['dist_min_ci_0_5h'] * X_train['closing_speed_m_per_h']
X_test['dist_speed_interaction'] = X_test['dist_min_ci_0_5h'] * X_test['closing_speed_m_per_h']

# 3. Growth × Speed interaction
# Captures: Overall fire expansion dynamics
# Fast-growing AND fast-moving fire = aggressive behavior
X_train['growth_speed_interaction'] = X_train['area_growth_rate_ha_per_h'] * X_train['centroid_speed_m_per_h']
X_test['growth_speed_interaction'] = X_test['area_growth_rate_ha_per_h'] * X_test['centroid_speed_m_per_h']

# 4. Distance × Alignment (directional threat)
# Captures: Is fire moving TOWARD the evacuation zone?
# alignment_abs measures how aligned fire movement is with zone direction
X_train['dist_alignment_interaction'] = X_train['dist_min_ci_0_5h'] * X_train['alignment_abs']
X_test['dist_alignment_interaction'] = X_test['dist_min_ci_0_5h'] * X_test['alignment_abs']

# 5. Area ÷ Distance ratio (threat density)
# Captures: Large fire nearby = high threat density
# Add 1 to denominator to avoid division by zero
# High value = Large fire close to zone
X_train['area_dist_ratio'] = X_train['area_first_ha'] / (X_train['dist_min_ci_0_5h'] + 1)
X_test['area_dist_ratio'] = X_test['area_first_ha'] / (X_test['dist_min_ci_0_5h'] + 1)

print("✅ Created 5 interaction features")
print(f"New feature count: {X_train.shape[1]}")
print(f"\n💡 These interactions capture wildfire threat physics:")
print(f"   - How fast is fire approaching?")
print(f"   - Is it moving toward the zone?")
print(f"   - How aggressive is its behavior?")

## 4. Create Domain-Specific Wildfire Features

### 🌲 Domain Knowledge in Feature Engineering

**Domain-specific features** leverage expert knowledge about wildfire behavior to create meaningful predictors.

**Key Concepts**:
1. **Risk Score**: Composite measure combining multiple threat factors
2. **Fire Intensity**: How aggressively is the fire burning?
3. **Threat Velocity**: Speed of fire moving toward zone
4. **Time to Impact**: Estimated hours until fire could reach zone
5. **Spread Efficiency**: How efficiently is fire expanding?

These features translate raw measurements into **actionable threat assessments**.

In [ ]:
# Composite Risk Score (normalized weighted combination)
def create_risk_score(df):
    """
    Create composite wildfire risk score.
    
    Combines multiple threat factors into a single 0-1 score:
    - Distance (30%): Closer fires are more threatening
    - Area (25%): Larger fires are more dangerous
    - Growth (25%): Fast-growing fires are aggressive
    - Speed (20%): Fast-moving fires give less warning
    
    All features normalized to 0-1 range for fair weighting.
    """
    # Normalize key features to 0-1 range
    # Add 1 to max to avoid division by zero
    area_norm = df['area_first_ha'] / (df['area_first_ha'].max() + 1)
    growth_norm = df['area_growth_rate_ha_per_h'] / (df['area_growth_rate_ha_per_h'].max() + 1)
    speed_norm = df['closing_speed_m_per_h'] / (df['closing_speed_m_per_h'].abs().max() + 1)
    
    # Distance: Inverse normalization (closer = higher risk)
    # 1 - normalized_distance means: far = 0, close = 1
    dist_norm = 1 - (df['dist_min_ci_0_5h'] / (df['dist_min_ci_0_5h'].max() + 1))
    
    # Weighted combination (weights sum to 1.0)
    risk_score = (0.3 * dist_norm +      # Distance is most important
                  0.25 * area_norm +      # Size matters
                  0.25 * growth_norm +    # Growth rate matters
                  0.2 * speed_norm)       # Speed matters
    return risk_score

X_train['wildfire_risk_score'] = create_risk_score(X_train)
X_test['wildfire_risk_score'] = create_risk_score(X_test)

print("✅ Created wildfire risk score")
print(f"Risk score statistics:\n{X_train['wildfire_risk_score'].describe()}")
print(f"\n💡 Risk score ranges from 0 (low threat) to 1 (high threat)")

In [ ]:
# 1. Fire Intensity (area × growth rate)
# Captures: How aggressively is the fire burning?
# Large AND fast-growing = very intense fire
X_train['fire_intensity'] = X_train['area_first_ha'] * X_train['area_growth_rate_ha_per_h']
X_test['fire_intensity'] = X_test['area_first_ha'] * X_test['area_growth_rate_ha_per_h']

# 2. Threat Velocity (closing speed × alignment)
# Captures: How fast is fire moving TOWARD the zone?
# High alignment + high speed = direct threat
X_train['threat_velocity'] = X_train['closing_speed_m_per_h'] * X_train['alignment_abs']
X_test['threat_velocity'] = X_test['closing_speed_m_per_h'] * X_test['alignment_abs']

# 3. Time to Potential Impact (distance ÷ speed)
# Captures: Estimated hours until fire could reach zone
# Add 1 to denominator to handle zero/low speeds safely
# Low value = fire could arrive soon
X_train['time_to_impact_est'] = X_train['dist_min_ci_0_5h'] / (X_train['closing_speed_abs_m_per_h'] + 1)
X_test['time_to_impact_est'] = X_test['dist_min_ci_0_5h'] / (X_test['closing_speed_abs_m_per_h'] + 1)

# 4. Fire Spread Efficiency (radial growth ÷ area)
# Captures: How efficiently is fire expanding?
# High value = fire spreading efficiently relative to its size
# Small but fast-spreading fires can be dangerous
X_train['spread_efficiency'] = X_train['radial_growth_rate_m_per_h'] / (X_train['area_first_ha'] + 1)
X_test['spread_efficiency'] = X_test['radial_growth_rate_m_per_h'] / (X_test['area_first_ha'] + 1)

print("✅ Created 4 domain-specific features")
print(f"New feature count: {X_train.shape[1]}")
print(f"\n💡 These features translate measurements into threat assessments:")
print(f"   - Fire intensity: How aggressive?")
print(f"   - Threat velocity: How fast approaching?")
print(f"   - Time to impact: When could it arrive?")
print(f"   - Spread efficiency: How efficiently expanding?")

## 5. Create Polynomial Features for Key Predictors

### 📐 Why Polynomial Features?

**Polynomial features** capture non-linear relationships by creating squared terms.

**Example**: Distance might have a non-linear effect:
- Fire at 1km vs 2km: Big difference in threat
- Fire at 10km vs 11km: Small difference in threat
- Squared term (distance²) captures this accelerating effect

**When to use**:
- Features with non-linear relationships to target
- Physical quantities that compound (area, speed, growth)
- Only apply to most important features (avoid feature explosion)

**Caution**: Creates many features, so we only square the top predictors from EDA.

In [ ]:
# Select top features for polynomial expansion (based on EDA)
# These showed strongest correlations with target in exploratory analysis
key_features = [
    'dist_min_ci_0_5h',           # Distance: Non-linear threat decay
    'area_first_ha',               # Area: Larger fires compound threat
    'closing_speed_m_per_h',       # Speed: Acceleration matters
    'area_growth_rate_ha_per_h'    # Growth: Exponential expansion
]

# Create squared terms to capture non-linear relationships
# x² emphasizes larger values more than smaller ones
# Example: distance=2 → squared=4, distance=10 → squared=100
for feat in key_features:
    X_train[f'{feat}_squared'] = X_train[feat] ** 2
    X_test[f'{feat}_squared'] = X_test[feat] ** 2

print(f"✅ Created {len(key_features)} squared features")
print(f"New feature count: {X_train.shape[1]}")
print(f"\n💡 Squared terms help models capture non-linear effects:")
print(f"   - Close fires are MUCH more threatening (distance²)")
print(f"   - Large fires compound their threat (area²)")
print(f"   - Fast growth accelerates danger (growth²)")

## 6. Address Multicollinearity

### 🔗 Why Remove Highly Correlated Features?

**Multicollinearity** occurs when features are highly correlated with each other.

**Problems it causes**:
1. **Redundant information**: Two features tell the same story
2. **Unstable models**: Small data changes cause large coefficient changes
3. **Harder interpretation**: Can't tell which feature is truly important
4. **Wasted computation**: Processing duplicate information

**Example**: `area_first_ha` and `area_last_ha` might be 95% correlated
- They provide nearly identical information
- Keeping both doesn't help the model
- Better to keep just one

**Strategy**: For each highly correlated pair (r > 0.9):
- Keep the feature more correlated with target
- Remove the less predictive one

In [ ]:
# Calculate correlation matrix for original features
# Focus on original features (not engineered ones) to avoid removing new features
original_features = [col for col in feature_cols if col in X_train.columns]
corr_matrix = X_train[original_features].corr().abs()

# Find highly correlated pairs (>0.9)
# Threshold of 0.9 means features share >81% of variance
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            high_corr_pairs.append({
                'feature1': corr_matrix.columns[i],
                'feature2': corr_matrix.columns[j],
                'correlation': corr_matrix.iloc[i, j]
            })

print(f"Found {len(high_corr_pairs)} highly correlated pairs (>0.9)")
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    print("\nTop 10 highly correlated pairs:")
    print(high_corr_df.sort_values('correlation', ascending=False).head(10))
else:
    print("✅ No highly correlated pairs found")

In [ ]:
# Remove one feature from each highly correlated pair
# Strategy: Keep the feature with higher correlation to target
features_to_remove = set()

if high_corr_pairs:
    # Calculate correlation with target (event)
    # This tells us which feature is more predictive
    target_corr = X_train[original_features].corrwith(y_train['event']).abs()
    
    for pair in high_corr_pairs:
        feat1, feat2 = pair['feature1'], pair['feature2']
        
        # Remove feature with lower target correlation
        # Keep the more predictive one
        if target_corr[feat1] < target_corr[feat2]:
            features_to_remove.add(feat1)
        else:
            features_to_remove.add(feat2)

print(f"\nFeatures to remove due to multicollinearity: {len(features_to_remove)}")
if features_to_remove:
    print(f"Removing: {list(features_to_remove)[:10]}...")  # Show first 10
    
    # Remove from both train and test
    X_train = X_train.drop(columns=list(features_to_remove))
    X_test = X_test.drop(columns=list(features_to_remove))
    
    print(f"✅ New feature count: {X_train.shape[1]}")
    print(f"\n💡 Removed redundant features while keeping most predictive ones")
else:
    print("✅ No features need removal")

## 7. Feature Scaling

### ⚖️ Why Scale Features?

**Feature scaling** ensures all features are on similar scales.

**Problem without scaling**:
- Distance: 0-50,000 meters
- Growth rate: 0-100 hectares/hour
- Models treat large numbers as more important

**Why RobustScaler?**
- **StandardScaler**: Uses mean and std (sensitive to outliers)
- **MinMaxScaler**: Uses min and max (very sensitive to outliers)
- **RobustScaler**: Uses median and IQR (robust to outliers) ✅

**How RobustScaler works**:
```
scaled_value = (value - median) / IQR
```
- Centers data around 0
- Outliers don't distort the scaling

**Important**: Don't scale binary features (already 0/1)

In [ ]:
# Identify features to scale (exclude binary indicators)
# Binary features (0/1) are already on the same scale
binary_features = [col for col in X_train.columns if col.endswith('_is_zero') or 
                   X_train[col].nunique() == 2]
features_to_scale = [col for col in X_train.columns if col not in binary_features]

print(f"Binary features (not scaled): {len(binary_features)}")
print(f"Features to scale: {len(features_to_scale)}")
print(f"\n💡 Binary features already on 0-1 scale, no scaling needed")

In [ ]:
# Apply RobustScaler (handles outliers better than StandardScaler)
scaler = RobustScaler()

# Create copies to avoid modifying original data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit scaler on training data only (prevent data leakage)
# Then transform both train and test using training statistics
X_train_scaled[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test_scaled[features_to_scale] = scaler.transform(X_test[features_to_scale])

print("✅ Scaling complete")
print(f"\nScaled feature statistics (first 5 features):")
print(X_train_scaled[features_to_scale[:5]].describe())
print(f"\n💡 Notice scaled features are centered around 0 with similar ranges")

## 8. Feature Selection

### 🎯 Why Feature Selection?

**Feature selection** identifies the most predictive features and removes the rest.

**Benefits**:
1. **Faster training**: Fewer features = faster models
2. **Better generalization**: Removes noise and irrelevant features
3. **Easier interpretation**: Focus on what matters
4. **Reduced overfitting**: Less chance to memorize noise

**Two Methods**:
1. **Correlation**: Linear relationship with target
   - Fast and interpretable
   - Only captures linear relationships

2. **Mutual Information (MI)**: Any relationship with target
   - Captures non-linear relationships
   - More comprehensive but slower

**Strategy**: Take union of top features from both methods
- Gets best of both worlds
- Ensures we don't miss important features

In [ ]:
# Method 1: Correlation-based selection
# Measures linear relationship between feature and target
# abs() because negative correlation is also informative
target_corr = X_train_scaled.corrwith(y_train['event']).abs().sort_values(ascending=False)

print("📊 Top 20 features by correlation with target:")
print(target_corr.head(20))

# Select top 50 features by correlation
top_corr_features = target_corr.head(50).index.tolist()
print(f"\n✅ Selected {len(top_corr_features)} features by correlation")

In [ ]:
# Method 2: Mutual Information
# Measures any relationship (linear or non-linear) between feature and target
# MI = 0: No relationship, MI > 0: Some relationship
print("\n🔍 Computing mutual information scores...")
mi_scores = mutual_info_classif(X_train_scaled, y_train['event'], random_state=RANDOM_STATE)
mi_scores_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print("\n📊 Top 20 features by mutual information:")
print(mi_scores_df.head(20))

# Select top 50 features by MI
top_mi_features = mi_scores_df.head(50)['feature'].tolist()
print(f"\n✅ Selected {len(top_mi_features)} features by mutual information")

In [ ]:
# Combine both methods - take union of top features
# Union ensures we capture both linear and non-linear relationships
selected_features = list(set(top_corr_features + top_mi_features))

print(f"\n🎯 Feature Selection Summary:")
print(f"="*50)
print(f"Total selected features: {len(selected_features)}")
print(f"Features from correlation only: {len(set(top_corr_features) - set(top_mi_features))}")
print(f"Features from MI only: {len(set(top_mi_features) - set(top_corr_features))}")
print(f"Features from both methods: {len(set(top_corr_features) & set(top_mi_features))}")
print(f"\n💡 Features selected by both methods are likely most important")

In [ ]:
# Create final feature sets
X_train_final = X_train_scaled[selected_features].copy()
X_test_final = X_test_scaled[selected_features].copy()

print(f"\n✅ Final feature set shape:")
print(f"Train: {X_train_final.shape}")
print(f"Test: {X_test_final.shape}")
print(f"\n💡 Reduced from {X_train_scaled.shape[1]} to {X_train_final.shape[1]} features")
print(f"   This will speed up training while maintaining predictive power")

## 9. Feature Importance Visualization

### 📊 Comparing Selection Methods

Visualizing feature importance from both methods helps us:
- See which features are consistently important
- Identify differences between linear and non-linear relationships
- Validate our feature selection choices

In [ ]:
# Visualize top features by both methods
# Side-by-side comparison helps identify consistent vs. method-specific features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: Correlation-based importance
# Shows features with strong linear relationships to target
target_corr.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Absolute Correlation with Target')
axes[0].set_title('Top 20 Features by Correlation\n(Linear Relationships)')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Right panel: Mutual Information importance
# Shows features with any relationship (linear or non-linear) to target
mi_scores_df.head(20).plot(x='feature', y='mi_score', kind='barh', ax=axes[1], 
                            color='coral', legend=False)
axes[1].set_xlabel('Mutual Information Score')
axes[1].set_title('Top 20 Features by Mutual Information\n(All Relationships)')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/feature_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Feature importance visualization saved")
print("\n💡 Compare the two panels:")
print("   - Features in both: Strong predictors (linear + non-linear)")
print("   - Only in correlation: Linear relationships")
print("   - Only in MI: Non-linear relationships")

## 10. Save Engineered Features

### 💾 Saving for Model Training

We save:
1. **Engineered features**: Ready for model training
2. **Target variables**: For training
3. **Feature names**: To track what was selected
4. **Scaler object**: To transform new data the same way

This ensures reproducibility and consistency.

In [ ]:
# Save processed features
# These are ready for model training - no further preprocessing needed
X_train_final.to_csv('../data/X_train_engineered.csv', index=False)
X_test_final.to_csv('../data/X_test_engineered.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)

# Save feature names for reference
# Useful for understanding which features were selected
pd.DataFrame({'feature': selected_features}).to_csv('../data/selected_features.csv', index=False)

# Save scaler for future use
# CRITICAL: Use same scaler on new data to maintain consistency
import joblib
joblib.dump(scaler, '../models/feature_scaler.pkl')

print("✅ Engineered features saved successfully")
print(f"\n📁 Files created:")
print("- ../data/X_train_engineered.csv (training features)")
print("- ../data/X_test_engineered.csv (test features)")
print("- ../data/y_train.csv (target variables)")
print("- ../data/selected_features.csv (feature names)")
print("- ../models/feature_scaler.pkl (scaler object)")
print("\n💡 These files are ready for the modeling notebook!")

## 11. Feature Engineering Summary

### 📋 Complete Pipeline Overview

This section summarizes the entire feature engineering pipeline:
- What we started with
- What we created
- What we removed
- What we ended with

Understanding this pipeline is crucial for:
- Reproducing results
- Applying to new data
- Explaining model inputs

In [ ]:
# Create comprehensive summary report
# This tracks every step of the feature engineering pipeline
summary = {
    'Original Features': len(feature_cols),
    'Zero Indicators Added': len(zero_inflated_features),
    'Log Transforms Added': len(zero_inflated_features),
    'Interaction Features': 5,
    'Domain-Specific Features': 4,  # risk_score + 4 others
    'Polynomial Features': len(key_features),
    'Features Removed (Multicollinearity)': len(features_to_remove) if features_to_remove else 0,
    'Final Selected Features': len(selected_features),
    'Feature Reduction': f"{(1 - len(selected_features)/X_train_scaled.shape[1])*100:.1f}%"
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("\n" + "="*60)
print("🎯 FEATURE ENGINEERING SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

# Save summary for documentation
summary_df.to_csv('../data/feature_engineering_summary.csv', index=False)
print("\n✅ Summary saved to ../data/feature_engineering_summary.csv")

In [ ]:
# Display sample of engineered features
# This gives a preview of what the model will see
print("\n📊 Sample of engineered features (first 5 rows, first 10 features):")
print(X_train_final.iloc[:5, :10])

print("\n📈 Feature statistics:")
print(X_train_final.describe())
print("\n💡 Notice:")
print("   - Features are scaled (centered around 0)")
print("   - Similar ranges across features")
print("   - Ready for model training!")

## 🎓 Key Insights & Next Steps

### What We Accomplished:

1. **Zero-Inflated Features** 🔢
   - Created binary indicators for presence/absence
   - Applied log1p transforms to handle skewness
   - Preserved information from sparse features

2. **Interaction Features** 🔥
   - Generated 5 physics-based interactions
   - Captured wildfire threat dynamics
   - Combined distance, growth, speed, and alignment

3. **Domain-Specific Features** 🌲
   - Created composite risk score
   - Engineered fire intensity and threat velocity
   - Translated measurements into threat assessments

4. **Polynomial Features** 📐
   - Added squared terms for key predictors
   - Captured non-linear relationships
   - Emphasized extreme values appropriately

5. **Multicollinearity Handling** 🔗
   - Removed redundant features (r > 0.9)
   - Kept most predictive features
   - Reduced model instability

6. **Feature Scaling** ⚖️
   - Applied RobustScaler for outlier resistance
   - Normalized feature ranges
   - Prepared for distance-based algorithms

7. **Feature Selection** 🎯
   - Combined correlation and mutual information
   - Selected most predictive features
   - Reduced dimensionality while maintaining power

### Ready for Modeling! 🚀

The engineered feature set is now:
- ✅ **Scaled**: All features on similar ranges
- ✅ **Selected**: Only most predictive features retained
- ✅ **Enhanced**: Rich with domain knowledge and interactions
- ✅ **Clean**: No multicollinearity or redundancy
- ✅ **Saved**: Ready for the modeling notebook

### Next Notebook: Modeling & Evaluation

We'll use these engineered features to:
1. Train survival analysis models (Cox PH, Random Survival Forests)
2. Generate multi-horizon probability forecasts (12h, 24h, 48h, 72h)
3. Evaluate using Hybrid Score (C-index + Weighted Brier Score)
4. Ensure monotonicity constraints
5. Analyze feature importance and model behavior